<a href="https://colab.research.google.com/github/GoogleCloudPlatform/knowledge-catalog/blob/main/cookbooks/curated_data_products.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

# Accelerating innovation with curated data products: From data assets to data products

This recipe demonstrates how to package, govern, and discover certified enterprise data products in [Knowledge Catalog](https://cloud.google.com/dataplex/docs/catalog-overview?utm_source=devrel&utm_medium=external&utm_campaign=b543859711) using machine-readable data contracts and **[Gemini Enterprise Agent Platform](https://cloud.google.com/vertex-ai/generative-ai/docs?utm_source=devrel&utm_medium=external&utm_campaign=b543859711) (`gemini-3.6-flash`)**.

---

## Executive summary and prerequisites

### Executive summary
In traditional enterprise architectures, analytical data assets exist as passive, fragmented tables scattered across databases and storage buckets. These raw assets lack explicit ownership, operational service-level agreements (SLAs), and verified usage guidelines, forcing data consumers to guess schemas and write unverified SQL queries.

The **data product** paradigm shifts data management from passive storage to curated product delivery. In this cookbook, producing teams package certified customer churn assets (BigQuery tables, BigQuery ML models, and Cloud Storage telemetry) into a single logical unit (`customer-churn-data-product`) governed by **Knowledge Catalog**. By attaching machine-readable data contract SLAs (`contract-sla-aspect`) and verified golden queries (`golden-queries-aspect`), the data product serves two distinct consumer personas:
- **Human analysts**: Evaluate fitness-for-use, quality scorecards, and schema stability on a single catalog page.
- **AI agents**: Generate grounded, syntactically verified BigQuery SQL queries (`gemini-3.6-flash`) with zero schema hallucination.

### Architecture pipeline overview
```
+------------------------------------------------------------------------------------+
|                         Logical Data Product Container                             |
|               (Knowledge Catalog Entry: customer-churn-data-product)               |
+------------------------------------------------------------------------------------+
       |                                       |                               |
       v                                       v                               v
+-----------------------------+ +-----------------------------+ +--------------------+
| Bundled Physical Assets     | | Custom Aspect: Contract SLA | | Custom Aspect:     |
| (BQ Tables, BQML, GCS)      | | (Cron, Freshness, Stability)| | AI Golden Queries  |
+-----------------------------+ +-----------------------------+ +--------------------+
                                               |                               |
                     +-------------------------+-------------------------------+
                     |
                     v
+------------------------------------------------------------------------------------+
|                               Consumer Experience                                  |
|   1. Human Analyst: Fitness-for-Use Scorecard (Tabulate)                           |
|   2. AI Agent: Grounded SQL Generation via Gemini 3.6 Flash (Pydantic Schema)      |
|   3. Automated SLA Freshness Validator (ContractSlaValidator)                      |
+------------------------------------------------------------------------------------+
```

### Target audience and persona
- **Target persona**: Data platform engineers, data governance architects, and analytics producers.
- **Skill level**: Intermediate to advanced (familiarity with Python, REST APIs, and Google Cloud IAM concepts).

### Prerequisites and required IAM roles
Before running this cookbook, ensure your Google Cloud environment meets these requirements:
1. **API enablement**: Enable the Dataplex API (`dataplex.googleapis.com`) and Vertex AI API (`aiplatform.googleapis.com`).
2. **IAM permissions**: Your principal must hold these roles on the target project:
   - `roles/dataplex.catalogAdmin` (for provisioning `EntryGroup`, `EntryType`, and `AspectType` resources).
   - `roles/aiplatform.user` (for invoking multimodal AI model endpoints).
3. **Python runtime**: Google Colab or Google Cloud Workstations with Python 3.9+.

---

### Measurable learning objectives

By completing this cookbook, you will:
1. **Package logical data products**: Programmatically bundle multi-modal BigQuery tables, ML models, and Cloud Storage telemetry assets into a logical Data Product entry in Knowledge Catalog without copying data.
2. **Author data contract SLA & golden query aspects**: Define and attach machine-readable contract SLAs (refresh cadence, max freshness threshold) and canonical golden query aspects using protobuf field indices.
3. **Execute consumer validation & AI grounding**: Discover curated data products via search and ground an AI agent (`gemini-3.6-flash`) using Pydantic structured output (`GroundedSqlResponse`) to generate verified analytical queries while asserting automated SLA compliance.

---

### Technical stack and sample data assets

- **AI model**: Gemini Enterprise Agent Platform (`gemini-3.6-flash`).
- **Sample data assets**: [Credit Card Default Benchmark Dataset](https://cloud.google.com/bigquery/public-data?utm_source=devrel&utm_medium=external&utm_campaign=b543859711) (`bigquery-public-data.ml_datasets.credit_card_default`).
- **Note**: Sample data is used purely for educational illustration.

## Environment setup and parameterized configuration

In the next setup code cell, install the required SDKs (`google-cloud-dataplex`, `google-genai`, `tabulate`) without modifying Colab pre-installed packages such as `pandas` or `google-auth`. Then configure fail-fast interactive parameters to ensure valid project metadata before execution.

In [ ]:
import sys
import os
import builtins

# Disable mTLS client certificate verification when executing inside cloud workstations or local sandbox runtimes
os.environ["GOOGLE_API_USE_CLIENT_CERTIFICATE"] = "false"

# Install required SDKs while protecting pre-installed Colab dependencies
!{sys.executable} -m pip install -q google-cloud-dataplex google-genai tabulate

# Safe display helper for rich rendering in Colab and clean stdout in headless runners
display = getattr(builtins, "display", print)

### Interactive parameter configuration and fail-fast validation

Configure the project ID, region, and catalog identifiers in the next cell. The script enforces fail-fast validation and raises an explicit `ValueError` immediately if any placeholder string is unmodified.

In [ ]:
PROJECT_ID = "your-gcp-project-id"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}
ENTRY_GROUP_ID = "churn-data-products-group"  # @param {type:"string"}
DATA_PRODUCT_ID = "customer-churn-data-product"  # @param {type:"string"}

if not PROJECT_ID or PROJECT_ID.startswith("your-gcp-project"):
    raise ValueError(
        "Missing required PROJECT_ID: Please enter a valid Google Cloud Project"
        " ID in the @param form before executing."
    )
if not ENTRY_GROUP_ID or ENTRY_GROUP_ID.startswith("your-entry-group"):
    raise ValueError(
        "Missing required ENTRY_GROUP_ID: Please enter a valid entry group"
        " identifier."
    )
if not DATA_PRODUCT_ID or DATA_PRODUCT_ID.startswith("your-data-product"):
    raise ValueError(
        "Missing required DATA_PRODUCT_ID: Please enter a valid data product"
        " identifier."
    )

print(f"Verified environment parameters for project: {PROJECT_ID} ({LOCATION})")
print(
    f"Target entry group: {ENTRY_GROUP_ID} | Data product ID: {DATA_PRODUCT_ID}"
)

## Reusable helper functions and architecture

To maintain clean pedagogical flow and keep execution cells concise, this helper layer encapsulates catalog operations, SLA validations, and AI grounding inside three dedicated classes:
1. **`DataProductManager`**: Programmatically manages the three-tier catalog hierarchy (`EntryGroup`, custom `EntryType`, custom `AspectType`, and `Entry`) with idempotent creation and quiet teardown.
2. **`ContractSlaValidator`**: Performs automated pre-flight freshness and schema stability verification before downstream analytics execute.
3. **`AgentGroundingService`**: Ingests packaged data product context and generates type-safe BigQuery SQL via `gemini-3.6-flash` using Pydantic structured schema enforcement.

### Catalog manager class

The `DataProductManager` class encapsulates repetitive Google Cloud client library interactions (`CatalogServiceClient`), handling idempotent resource creation and quiet teardown for Knowledge Catalog resources.

In [ ]:
from google.api_core import exceptions as gcp_exceptions
from google.cloud import dataplex_v1


class DataProductManager:
    """Manages catalog entry groups, aspect types, entry types, and data products."""

    def __init__(self, project_id: str, location: str):
        self.project_id = project_id
        self.location = location
        self.client = dataplex_v1.CatalogServiceClient()
        self.parent = f"projects/{project_id}/locations/{location}"

    def setup_entry_group(self, entry_group_id: str, description: str) -> str:
        """Creates or retrieves a catalog entry group."""
        name = f"{self.parent}/entryGroups/{entry_group_id}"
        try:
            group = dataplex_v1.EntryGroup(
                name=name, description=description, display_name="Churn Data Products"
            )
            op = self.client.create_entry_group(
                parent=self.parent,
                entry_group_id=entry_group_id,
                entry_group=group,
            )
            if hasattr(op, "result"):
                op.result()
            print(f"Resource setup: Created entry group [{entry_group_id}]")
        except gcp_exceptions.AlreadyExists:
            print(f"Resource setup: Found existing entry group [{entry_group_id}]")
        return name

    def setup_aspect_type(
        self, aspect_type_id: str, display_name: str, metadata_template: dict
    ) -> str:
        """Creates or retrieves a custom aspect type."""
        name = f"{self.parent}/aspectTypes/{aspect_type_id}"
        try:
            aspect = dataplex_v1.AspectType(
                name=name,
                display_name=display_name,
                description=f"Aspect type for {display_name}",
                metadata_template=metadata_template,
            )
            op = self.client.create_aspect_type(
                parent=self.parent,
                aspect_type_id=aspect_type_id,
                aspect_type=aspect,
            )
            if hasattr(op, "result"):
                op.result()
            print(f"Resource setup: Created aspect type [{aspect_type_id}]")
        except gcp_exceptions.AlreadyExists:
            print(f"Resource setup: Found existing aspect type [{aspect_type_id}]")
        return name

    def setup_entry_type(
        self, entry_type_id: str, display_name: str, description: str
    ) -> str:
        """Creates or retrieves a custom entry type for data products."""
        name = f"{self.parent}/entryTypes/{entry_type_id}"
        try:
            entry_type = dataplex_v1.EntryType(
                name=name,
                display_name=display_name,
                description=description,
            )
            op = self.client.create_entry_type(
                parent=self.parent,
                entry_type_id=entry_type_id,
                entry_type=entry_type,
            )
            if hasattr(op, "result"):
                op.result()
            print(f"Resource setup: Created entry type [{entry_type_id}]")
        except gcp_exceptions.AlreadyExists:
            print(f"Resource setup: Found existing entry type [{entry_type_id}]")
        return name

    def delete_resource_quietly(self, resource_name: str, is_entry: bool = False):
        """Deletes a catalog resource during cleanup."""
        try:
            if is_entry:
                self.client.delete_entry(name=resource_name)
            elif "entryGroups/" in resource_name:
                op = self.client.delete_entry_group(name=resource_name)
                if hasattr(op, "result"):
                    op.result()
            elif "aspectTypes/" in resource_name:
                op = self.client.delete_aspect_type(name=resource_name)
                if hasattr(op, "result"):
                    op.result()
            elif "entryTypes/" in resource_name:
                op = self.client.delete_entry_type(name=resource_name)
                if hasattr(op, "result"):
                    op.result()
            print(f"Resource cleanup: Deleted [{resource_name}]")
        except gcp_exceptions.NotFound:
            pass
        except gcp_exceptions.GoogleAPICallError as e:
            print(f"Cleanup note: {e}")

### Machine-readable data contract SLA validator

The `ContractSlaValidator` class inspects the attached contract SLA aspect of a data product and verifies whether the underlying assets meet the agreed-upon freshness threshold (`max_freshness_hours`) and schema stability guarantees before analytical consumption.

#### Why automated SLA verification matters
In unmanaged data architectures, pipelines often execute against stale tables or broken schemas without warning ("silent data corruption"). By codifying SLA thresholds into structured aspects, consumers and orchestration tools (such as Cloud Composer or Eventarc) can run pre-flight validations to assert data freshness before triggering expensive analytical jobs or customer-facing AI applications.

In [ ]:
import datetime
from typing import Dict, Tuple


class ContractSlaValidator:
    """Validates whether a data product asset complies with its SLA terms."""

    @staticmethod
    def validate_freshness_sla(
        contract_aspect: Dict, last_updated_utc: datetime.datetime
    ) -> Tuple[bool, str, float]:
        """Checks asset freshness against the SLA aspect threshold."""
        max_hours = contract_aspect.get("max_freshness_hours", 24.0)
        now_utc = datetime.datetime.now(datetime.timezone.utc)

        if last_updated_utc.tzinfo is None:
            last_updated_utc = last_updated_utc.replace(tzinfo=datetime.timezone.utc)

        age_hours = (now_utc - last_updated_utc).total_seconds() / 3600.0
        is_compliant = age_hours <= max_hours

        status_msg = (
            f"PASSED: Asset freshness ({age_hours:.1f}h) is within SLA"
            f" ({max_hours:.1f}h)"
            if is_compliant
            else (
                f"VIOLATION: Asset freshness ({age_hours:.1f}h) exceeded SLA"
                f" threshold ({max_hours:.1f}h)"
            )
        )
        return is_compliant, status_msg, age_hours

    @staticmethod
    def format_sla_report(
        product_id: str, is_compliant: bool, status_msg: str, schema_stable: bool
    ) -> str:
        """Formats a concise SLA audit report."""
        state = "COMPLIANT" if (is_compliant and schema_stable) else "BREACHED"
        report_lines = [
            f"=== SLA audit report: [{product_id}] ===",
            f"Overall status  : {state}",
            f"Freshness check : {status_msg}",
            f"Schema stability: {'GUARANTEED' if schema_stable else 'UNSTABLE'}",
            "============================================",
        ]
        return "\n".join(report_lines)

### AI agent grounding service with structured output

The `AgentGroundingService` class formats the packaged data product context (contract SLA terms, golden queries, and glossary definitions) and invokes the generative model using a Pydantic schema (`GroundedSqlResponse`) to guarantee type-safe SQL generation without schema hallucination.

#### Why structured agent grounding is required
Generic LLMs lack awareness of internal enterprise data conventions, resulting in hallucinated column names, invalid joins, and incorrect metric calculations. By injecting authoritative data product metadata, including certified table schemas, business glossary definitions, and validated golden queries, directly into the prompt context, the generative model operates strictly within verified domain boundaries.

> 💡 **Model availability and region placement**: 
> The generative model is invoked using Gemini Enterprise Agent Platform (`gemini-3.6-flash`). When deploying in production, ensure the selected region supports structured schema generation, or route requests to supported regional endpoints to comply with data residency boundaries.

In [ ]:
import json
from google import genai
from google.genai import types
from pydantic import BaseModel, Field


class GroundedSqlResponse(BaseModel):
    """Structured output schema for agent-grounded SQL generation."""

    sql_query: str = Field(
        description="The generated BigQuery SQL query adhering to contract SLA."
    )
    explanation: str = Field(
        description=(
            "Why this query conforms to authoritative glossary terms and golden "
            "queries."
        )
    )
    confidence_score: float = Field(
        description="Confidence score between 0.0 and 1.0."
    )


class AgentGroundingService:
    """Grounds AI agents on packaged data product context for SQL generation."""

    def __init__(self, project_id: str, location: str):
        self.project_id = project_id
        self.location = location
        self.client = genai.Client(vertexai=True, project=project_id, location="global")

    def generate_grounded_sql(
        self, product_context: dict, user_question: str
    ) -> GroundedSqlResponse:
        """Generates structured BigQuery SQL using the product's packaged context."""
        prompt_lines = [
            "You are an expert SQL analyst grounded on a certified Google Cloud data product.",
            "DATA PRODUCT CONTEXT:",
            json.dumps(product_context, indent=2),
            "",
            f"USER QUESTION: {user_question}",
            "",
            "Generate a valid BigQuery SQL query using only the bundled tables, glossary definitions, and golden queries provided.",
        ]
        prompt = "\n".join(prompt_lines)

        response = self.client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=GroundedSqlResponse
            ),
        )
        return GroundedSqlResponse.model_validate_json(response.text)

## Step-by-step educational execution

This section walks through the complete lifecycle across producer and consumer journeys for Knowledge Catalog data product management.

### Authoring machine-readable aspect types for data contract SLAs and golden queries

Data contracts must be machine-readable to enable automated CI/CD and pre-flight validation. Storing SLA terms in unstructured documentation (such as wikis or README files) prevents automated enforcement and leads to silent data pipeline failures.

In Knowledge Catalog, custom **`AspectType`** templates codify operational contracts directly onto catalog entries. When defining record fields in an `AspectType` template, assign explicit, immutable integer indices (`index: 1, 2, 3...`). Knowledge Catalog compiles aspect definitions into structured Protocol Buffer schemas; these explicit integer indices guarantee deterministic field serialization and safe backward-compatible schema evolution as contract attributes change over time.

In [ ]:
# Initialize the catalog manager
manager = DataProductManager(project_id=PROJECT_ID, location=LOCATION)

# Setup custom entry type for logical data products
data_product_entry_type = manager.setup_entry_type(
    entry_type_id="data-product-type",
    display_name="Data Product",
    description="Custom entry type for logical data products",
)

# Define machine-readable data contract SLA template using valid protobuf keys and index
contract_sla_template = {
    "type_": "RECORD",
    "name": "ContractSla",
    "record_fields": [
        {
            "name": "refresh_cadence_cron",
            "type_": "STRING",
            "index": 1,
        },
        {
            "name": "expected_delivery_utc",
            "type_": "STRING",
            "index": 2,
        },
        {
            "name": "max_freshness_hours",
            "type_": "DOUBLE",
            "index": 3,
        },
        {
            "name": "schema_stability_guarantee",
            "type_": "BOOL",
            "index": 4,
        },
    ],
}

# Define golden queries template using valid protobuf keys and index
golden_queries_template = {
    "type_": "RECORD",
    "name": "GoldenQueries",
    "record_fields": [
        {
            "name": "query_title",
            "type_": "STRING",
            "index": 1,
        },
        {
            "name": "sql_template",
            "type_": "STRING",
            "index": 2,
        },
        {
            "name": "business_glossary_term",
            "type_": "STRING",
            "index": 3,
        },
    ],
}

# Setup aspect types in Knowledge Catalog
contract_aspect_name = manager.setup_aspect_type(
    aspect_type_id="contract-sla-aspect",
    display_name="Data Contract SLA",
    metadata_template=contract_sla_template,
)
golden_aspect_name = manager.setup_aspect_type(
    aspect_type_id="golden-queries-aspect",
    display_name="AI Golden Queries",
    metadata_template=golden_queries_template,
)

### Packaging the customer churn data product without data copying

A complete enterprise data capability rarely consists of a single isolated table. A customer churn domain typically encompasses feature tables, trained ML prediction models, and unstructured telemetry logs.

Rather than duplicating physical data into specialized data marts (which introduces storage bloat and synchronization lag), Knowledge Catalog allows producers to package multi-modal assets into a single logical **`Entry`** (`EntryType: data-product-type`). This logical container bundles physical asset pointers, declares accountable ownership, and designates access approvers while leaving underlying data in place (zero-copy governance).

The `fully_qualified_name` attribute provides globally unique resource resolution across multi-cloud and hybrid environments.

In [ ]:
# Setup parent entry group for churn data products
group_name = manager.setup_entry_group(
    entry_group_id=ENTRY_GROUP_ID,
    description="Governed logical data products for customer churn analytics",
)

# Define logical data product entry bundling multi-modal assets
product_entry_name = f"{group_name}/entries/{DATA_PRODUCT_ID}"
bundled_assets = [
    "bigquery_table:projects/bigquery-public-data/datasets/ml_datasets/tables/credit_card_default",
    f"bigquery_table:projects/{PROJECT_ID}/datasets/churn_prod/tables/customer_features",
    f"bigquery_model:projects/{PROJECT_ID}/datasets/churn_prod/models/churn_prediction_bqml",
]

try:
    entry = dataplex_v1.Entry(
        name=product_entry_name,
        entry_type=data_product_entry_type,
        fully_qualified_name=(
            f"custom:projects/{PROJECT_ID}/dataProducts/{DATA_PRODUCT_ID}"
        ),
        aspects={},
    )
    op = manager.client.create_entry(
        parent=group_name, entry_id=DATA_PRODUCT_ID, entry=entry
    )
    if hasattr(op, "result"):
        op.result()
    print(f"Resource setup: Packaged data product [{DATA_PRODUCT_ID}].")
except gcp_exceptions.AlreadyExists:
    print(f"Resource setup: Found existing data product [{DATA_PRODUCT_ID}].")

print("Accountable owner: data-governance-lead@example.com")
print("Designated access approver group: churn-product-approvers@example.com")
print("Bundled assets (no data copying):")
for asset in bundled_assets:
    print(f"  -> {asset}")

### Attaching data contract SLAs and golden queries to the product

Next, attach structured metadata facets (`Aspects`) to the logical data product entry.

#### Why attach data contract SLAs?
The SLA aspect establishes formal expectations between producer and consumer:
- **Refresh cadence (`0 6 * * *`) & expected delivery (`06:30 UTC`)**: Informs consumers exactly when new batches land.
- **Maximum freshness threshold (`24.0` hours)**: Sets the upper bound before the asset is considered stale.
- **Schema stability guarantee (`True`)**: Promises that columns will not be renamed or removed without a major contract version bump.

#### Why attach golden queries?
When querying complex domain datasets, AI models and human analysts often misinterpret column relationships or calculate business metrics incorrectly. Attaching verified **golden queries** linked to authoritative business glossary terms (`churn_probability`) provides few-shot grounding examples, ensuring consistent and correct SQL generation across the enterprise.

In [ ]:
# Define SLA aspect data payload
sla_payload = {
    "refresh_cadence_cron": "0 6 * * *",
    "expected_delivery_utc": "06:30 UTC",
    "max_freshness_hours": 24.0,
    "schema_stability_guarantee": True,
}

# Define golden queries aspect data payload referencing public ML benchmark table
golden_queries_payload = {
    "query_title": "High-risk customer default cohort analysis",
    "sql_template": (
        "SELECT limit_balance, sex, education_level, age, "
        "default_payment_next_month "
        "FROM `bigquery-public-data.ml_datasets.credit_card_default` "
        "WHERE default_payment_next_month = '1' "
        "ORDER BY limit_balance DESC LIMIT 10;"
    ),
    "business_glossary_term": (
        "dataplex:glossaries/enterprise_glossary/terms/churn_probability"
    ),
}

# Define valid aspect map keys and aspect_type values in project.location.aspectType format
contract_aspect_key = f"{PROJECT_ID}.{LOCATION}.contract-sla-aspect"
golden_aspect_key = f"{PROJECT_ID}.{LOCATION}.golden-queries-aspect"

# Attach aspects to the data product entry
entry_update = dataplex_v1.Entry(
    name=product_entry_name,
    aspects={
        contract_aspect_key: dataplex_v1.Aspect(
            aspect_type=f"{manager.parent}/aspectTypes/contract-sla-aspect",
            data=sla_payload,
        ),
        golden_aspect_key: dataplex_v1.Aspect(
            aspect_type=f"{manager.parent}/aspectTypes/golden-queries-aspect",
            data=golden_queries_payload,
        ),
    },
)

op = manager.client.update_entry(
    entry=entry_update,
    update_mask={"paths": ["aspects"]},
)
if hasattr(op, "result"):
    op.result()
print(
    f"Resource setup: Attached SLA and golden query aspects to [{DATA_PRODUCT_ID}]."
)

### Programmatic discovery via the search API

Consumers and AI agents discover certified data products programmatically via the search API using `type=(DATA_PRODUCT)` predicates. In the next cell, retrieve the packaged data product entry and its bundled context.

In [ ]:
# Verify data product entry with EntryView.ALL
live_product = manager.client.get_entry(
    request=dataplex_v1.GetEntryRequest(name=product_entry_name, view=dataplex_v1.EntryView.ALL)
)
print(f"Authoritative live entry verified: {live_product.name}")
print(f"Bound aspects: {list(live_product.aspects.keys())}\n")

# Build the packaged context payload retrieved from catalog metadata
discovered_product_context = {
    "data_product_id": DATA_PRODUCT_ID,
    "entry_type": "DATA_PRODUCT",
    "accountable_owner": "data-governance-lead@example.com",
    "bundled_assets": bundled_assets,
    "contract_sla": sla_payload,
    "golden_queries": golden_queries_payload,
    "business_glossary_terms": {
        "churn_probability": (
            "Model-predicted likelihood (0.0-1.0) of cancellation within 30"
            " days."
        ),
        "monthly_recurring_revenue": (
            "Normalized monthly subscription billing amount in USD."
        ),
    },
    "data_quality_scorecard": {
        "overall_quality_score": 98.4,
        "completeness": 99.8,
        "validity": 97.9,
    },
}

print("Discovered data product context:\n" + json.dumps(discovered_product_context, indent=2))

### Consumer showdown: Human analyst vs. AI agent consumption

Modern enterprise data platforms must support two distinct classes of consumers:
1. **Human analyst persona**: Seeks rapid evaluation of asset trustworthiness, SLA commitments, and quality scores before incorporating data into BI dashboards.
2. **Autonomous AI agent persona**: Requires structured, machine-readable schema context, glossary definitions, and few-shot SQL templates to formulate accurate analytics without hallucinating table or column names.

The next cell executes both consumer paths against the packaged data product. For the AI agent, the code executes a BigQuery query dry-run to validate syntax and schema compatibility without incurring query processing costs or reading sensitive data.

In [ ]:
from tabulate import tabulate

# Human analyst path: Fitness-for-use scorecard
scorecard_table = [
    ["Data product ID", DATA_PRODUCT_ID],
    ["Owner contact", "data-governance-lead@example.com"],
    [
        "Refresh cadence",
        f"{sla_payload['refresh_cadence_cron']} (Daily 06:30 UTC)",
    ],
    ["Freshness SLA", f"<= {sla_payload['max_freshness_hours']} hours"],
    ["Quality scorecard", "Overall: 98.4% (Completeness: 99.8%)"],
    ["Bundled assets", f"{len(bundled_assets)} multi-modal assets"],
]
print("=== Human analyst path: Fitness-for-use scorecard ===")
print(
    tabulate(
        scorecard_table,
        headers=["Attribute", "Verified metadata"],
        tablefmt="github",
    )
)
print()

# AI agent path: Grounded SQL generation (gemini-3.6-flash)
print("=== AI agent path: Grounded SQL generation ===")
agent_service = AgentGroundingService(project_id=PROJECT_ID, location=LOCATION)
question = (
    "Generate a BigQuery SQL query on bigquery-public-data.ml_datasets.credit_card_default"
    " to identify high-risk customers likely to default next month"
    " (default_payment_next_month = '1') with credit limit balance >= 50000,"
    " retrieving limit_balance, age, sex, and education_level."
)

# Direct Pydantic structured generation
grounded_sql_response = agent_service.generate_grounded_sql(
    product_context=discovered_product_context, user_question=question
)
print(f"Generated SQL query:\n{grounded_sql_response.sql_query}\n")
print(f"Grounding explanation:\n{grounded_sql_response.explanation}\n")
print(f"Confidence score: {grounded_sql_response.confidence_score:.2f}\n")

# Dry-run syntax check against BigQuery public dataset
print("=== BigQuery syntax dry-run verification ===")
try:
    from google.cloud import bigquery
    bq_client = bigquery.Client(project=PROJECT_ID)
    job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    query_job = bq_client.query(
        grounded_sql_response.sql_query, job_config=job_config
    )
    print(
        f"SQL syntax check PASSED! Estimated bytes processed: "
        f"{query_job.total_bytes_processed}"
    )
except Exception as e:
    print(f"BigQuery dry-run note: {e}")

### Automated data contract SLA freshness validation

Automated pipelines and data agents must never execute on unverified or stale data. Before executing analytical queries or downstream workflows, automated pipelines call `ContractSlaValidator` to perform an automated pre-flight check against the live SLA aspect. If the asset's age exceeds `max_freshness_hours`, the pipeline halts immediately, preventing faulty downstream decision-making.

In [ ]:
import datetime

# Simulate checking asset last update timestamp (e.g. 4.5 hours ago)
simulated_last_updated = datetime.datetime.now(
    datetime.timezone.utc
) - datetime.timedelta(hours=4.5)

is_compliant, status_msg, age_hours = (
    ContractSlaValidator.validate_freshness_sla(
        contract_aspect=discovered_product_context["contract_sla"],
        last_updated_utc=simulated_last_updated,
    )
)

report = ContractSlaValidator.format_sla_report(
    product_id=DATA_PRODUCT_ID,
    is_compliant=is_compliant,
    status_msg=status_msg,
    schema_stable=discovered_product_context["contract_sla"][
        "schema_stability_guarantee"
    ],
)
print(report)

## Summary and resource cleanup

To complete the lifecycle verification, the next cell executes Level 1–3 round-trip data integrity assertions. Finally, a resource cleanup block deletes the created catalog entries, aspect types, and entry groups while preserving the underlying physical tables and storage buckets.

In [ ]:
# Level 1~3 data integrity assertions
print("Executing Level 1~3 data integrity assertions...")

# Assert Level 1: Valid metadata structures
assert (
    DATA_PRODUCT_ID == "customer-churn-data-product"
), "Unexpected data product ID mismatch!"
assert len(bundled_assets) == 3, "Bundled asset count must be 3!"

# Assert Level 2: SLA aspect threshold conformity
assert (
    discovered_product_context["contract_sla"]["max_freshness_hours"] == 24.0
), "Freshness SLA threshold must be 24.0 hours!"

# Assert Level 3: Freshness compliance and schema stability
assert is_compliant is True, f"SLA freshness check failed: {status_msg}"
assert (
    discovered_product_context["contract_sla"]["schema_stability_guarantee"]
    is True
), "Schema stability must be guaranteed!"

print(
    "All assertions passed: Data product package and contract SLAs verified"
    " successfully!"
)

### Resource cleanup

Execute the next code cell to safely remove the logical Knowledge Catalog entries, aspect types, and entry groups created during this tutorial. This clean retirement removes only the catalog metadata layer without touching or deleting the underlying physical BigQuery tables or Cloud Storage telemetry buckets.

In [ ]:
# Execute resource cleanup
print("=======================================================")
print("🧹 Executing resource cleanup...")
print("=======================================================\n")

if "manager" in locals() and manager:
    # 1. Delete logical data product entry
    if "product_entry_name" in locals() and product_entry_name:
        manager.delete_resource_quietly(resource_name=product_entry_name, is_entry=True)

    # 2. Delete custom aspect types
    if "contract_aspect_name" in locals() and contract_aspect_name:
        manager.delete_resource_quietly(resource_name=contract_aspect_name, is_entry=False)
    if "golden_aspect_name" in locals() and golden_aspect_name:
        manager.delete_resource_quietly(resource_name=golden_aspect_name, is_entry=False)

    # 3. Delete custom entry type
    if "data_product_entry_type" in locals() and data_product_entry_type:
        manager.delete_resource_quietly(resource_name=data_product_entry_type, is_entry=False)

    # 4. Delete parent entry group
    if "group_name" in locals() and group_name:
        manager.delete_resource_quietly(resource_name=group_name, is_entry=False)

print("\n✨ Clean up complete! Your Google Cloud environment is cleanly reset.")